<a href="https://colab.research.google.com/github/M1NG0LL/Driver-Drowsiness-Detection-DDD/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Importing

In [2]:
import os
import kagglehub
import random
import numpy as np
import hashlib
import tempfile
from PIL import Image
import shutil
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Dense, Dropout, Conv2D, Input, GlobalAveragePooling2D,
    BatchNormalization, Activation, Concatenate, Add, MaxPooling2D
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.applications.efficientnet import preprocess_input
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    precision_recall_curve, f1_score
)
from collections import Counter
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# DataSet

installing dataset

In [3]:
data_dir = kagglehub.dataset_download('ismailnasri20/driver-drowsiness-dataset-ddd')

print('Dataset downloaded to:', data_dir)

Using Colab cache for faster access to the 'driver-drowsiness-dataset-ddd' dataset.
Dataset downloaded to: /kaggle/input/driver-drowsiness-dataset-ddd


In [4]:
!pip install datasets

In [7]:
from datasets import load_dataset
ds = load_dataset("n7i5x9/driver-drowsiness-dataset")
print(ds["train"][0])

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=640x640 at 0x7EF49D37AAE0>, 'label': 0}


In [8]:
# Print the number of items in each split
print(f"Number of training samples: {len(ds['train'])}")
print(f"Number of validation samples: {len(ds['validation'])}")
print(f"Number of test samples: {len(ds['test'])}")

# Access the first training sample
first_train_sample = ds['train'][0]

# Print the keys available in the sample
print(f"\nKeys in the first training sample: {first_train_sample.keys()}")

# Display the image and its label
print(f"Label of the first training sample: {first_train_sample['label']}")
# display(first_train_sample['image'])

Number of training samples: 18492
Number of validation samples: 2311
Number of test samples: 2313

Keys in the first training sample: dict_keys(['image', 'label'])
Label of the first training sample: 0


Split Data

In [ ]:
data_dir = '/kaggle/input/driver-drowsiness-dataset-ddd/Driver Drowsiness Dataset (DDD)'
output_dir = "Drowsy_Split"

classes = ["Drowsy", "Non Drowsy"]
seed = 42

print("Starting dataset merge + split")
print("Input folder:", data_dir)
print("Output folder:", output_dir)

def get_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

def hash_image(img):
    buffer = tempfile.SpooledTemporaryFile()
    img.save(buffer, format="PNG")
    buffer.seek(0)
    return hashlib.md5(buffer.read()).hexdigest()

data = []
seen_hashes = set()

print("\nScanning folder dataset...")

for label in classes:
    folder = os.path.join(data_dir, label)
    files = os.listdir(folder)

    print("Processing:", label, "| files:", len(files))

    for file in files:
        path = os.path.join(folder, file)

        try:
            h = get_hash(path)
        except:
            continue

        if h in seen_hashes:
            continue

        seen_hashes.add(h)
        data.append((path, label))

print("Folder samples:", len(data))

print("\nAdding HF dataset...")

hf_data = []

for split in ds.keys():
    for item in ds[split]:
        img = item["image"]
        label = item["label"]

        try:
            h = hash_image(img)
        except:
            continue

        if h in seen_hashes:
            continue

        seen_hashes.add(h)
        hf_data.append((img, label))

print("HF samples:", len(hf_data))

all_data = data + hf_data

print("\nTotal unique samples after merge:", len(all_data))

X = [x[0] for x in all_data]
y = [x[1] for x in all_data]

print("\nSplitting dataset...")

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=seed
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=seed
)

print("Train:", len(X_train))
print("Val:", len(X_val))
print("Test:", len(X_test))

print("\nCreating output folders...")

for split in ["train", "val", "test"]:
    for c in classes:
        path = os.path.join(output_dir, split, c)
        os.makedirs(path, exist_ok=True)

def copy_data(paths, labels, split):
    print("\nCopying", split, "set...")

    for i, (p, l) in enumerate(zip(paths, labels)):

        if isinstance(p, str):
            dst = os.path.join(output_dir, split, l, os.path.basename(p))
            shutil.copy2(p, dst)
        else:
            filename = f"{split}_{i}.png"
            dst = os.path.join(output_dir, split, l, filename)
            p.save(dst)

        if i % 200 == 0:
            print(i, "/", len(paths))

copy_data(X_train, y_train, "train")
copy_data(X_val, y_val, "val")
copy_data(X_test, y_test, "test")

print("\nDONE ✔ Dataset ready")
print("Location:", os.path.abspath(output_dir))

Starting dataset merge + split
Input folder: /kaggle/input/driver-drowsiness-dataset-ddd/Driver Drowsiness Dataset (DDD)
Output folder: Drowsy_Split

Scanning folder dataset...
Processing: Drowsy | files: 22348
Processing: Non Drowsy | files: 19445
Folder samples: 41793

Adding HF dataset...


In [5]:
# data_dir = '/kaggle/input/driver-drowsiness-dataset-ddd/Driver Drowsiness Dataset (DDD)'
# output_dir = "Drowsy_Split"

# classes = ["Drowsy", "Non Drowsy"]
# seed = 42

# print("Starting dataset split")
# print("Input folder:", data_dir)
# print("Output folder:", output_dir)


# def get_hash(path):
#     with open(path, "rb") as f:
#         return hashlib.md5(f.read()).hexdigest()


# # 1. Load + remove duplicates
# data = []
# seen_hashes = set()

# print("\nScanning dataset and removing duplicates...")

# for label in classes:
#     folder = os.path.join(data_dir, label)
#     files = os.listdir(folder)

#     print("Processing class:", label, "| files:", len(files))

#     for file in files:
#         path = os.path.join(folder, file)

#         try:
#             h = get_hash(path)
#         except:
#             print("Skipping corrupted file:", path)
#             continue

#         if h in seen_hashes:
#             continue

#         seen_hashes.add(h)
#         data.append((path, label))

# print("\nUnique images after deduplication:", len(data))


# paths = [x[0] for x in data]
# labels = [x[1] for x in data]


# # 2. Split
# print("\nSplitting dataset...")

# X_train, X_temp, y_train, y_temp = train_test_split(
#     paths, labels,
#     test_size=0.2,
#     stratify=labels,
#     random_state=seed
# )

# X_val, X_test, y_val, y_test = train_test_split(
#     X_temp, y_temp,
#     test_size=0.5,
#     stratify=y_temp,
#     random_state=seed
# )

# print("Train:", len(X_train), "| Val:", len(X_val), "| Test:", len(X_test))


# # 3. Create folders
# print("\nCreating output folders...")

# for split in ["train", "val", "test"]:
#     for c in classes:
#         path = os.path.join(output_dir, split, c)
#         os.makedirs(path, exist_ok=True)
#         print("Created:", path)


# # 4. Copy files
# def copy_data(paths, labels, split):
#     print("\nCopying", split, "set...")
#     for i, (p, l) in enumerate(zip(paths, labels)):
#         dst = os.path.join(output_dir, split, l, os.path.basename(p))
#         shutil.copy2(p, dst)

#         if i % 200 == 0:
#             print(i, "/", len(paths), "copied")


# copy_data(X_train, y_train, "train")
# copy_data(X_val, y_val, "val")
# copy_data(X_test, y_test, "test")

# print("\nDone. Dataset split completed.")
# print("Location:", os.path.abspath(output_dir))

Starting dataset split
Input folder: /kaggle/input/driver-drowsiness-dataset-ddd/Driver Drowsiness Dataset (DDD)
Output folder: Drowsy_Split

Scanning dataset and removing duplicates...
Processing class: Drowsy | files: 22348
Processing class: Non Drowsy | files: 19445

Unique images after deduplication: 41793

Splitting dataset...
Train: 33434 | Val: 4179 | Test: 4180

Creating output folders...
Created: Drowsy_Split/train/Drowsy
Created: Drowsy_Split/train/Non Drowsy
Created: Drowsy_Split/val/Drowsy
Created: Drowsy_Split/val/Non Drowsy
Created: Drowsy_Split/test/Drowsy
Created: Drowsy_Split/test/Non Drowsy

Copying train set...
0 / 33434 copied
200 / 33434 copied
400 / 33434 copied
600 / 33434 copied
800 / 33434 copied
1000 / 33434 copied
1200 / 33434 copied
1400 / 33434 copied
1600 / 33434 copied
1800 / 33434 copied
2000 / 33434 copied
2200 / 33434 copied
2400 / 33434 copied
2600 / 33434 copied
2800 / 33434 copied
3000 / 33434 copied
3200 / 33434 copied
3400 / 33434 copied
3600 / 33

In [6]:
train_dir = os.path.join(output_dir, "train")
val_dir   = os.path.join(output_dir, "val")
test_dir  = os.path.join(output_dir, "test")

In [10]:
def get_all_files(base_dir):
    all_files = set()

    for class_name in os.listdir(base_dir):  # Drowsy / Not_Drowsy
        class_path = os.path.join(base_dir, class_name)

        if os.path.isdir(class_path):
            for file in os.listdir(class_path):
                full_path = os.path.join(class_name, file)
                all_files.add(full_path)

    return all_files


train_files = get_all_files(train_dir)
val_files = get_all_files(val_dir)
test_files = get_all_files(test_dir)

print("Train-Val Overlap:", len(train_files & val_files))
print("Train-Test Overlap:", len(train_files & test_files))
print("Val-Test Overlap:", len(val_files & test_files))

Train-Val Overlap: 0
Train-Test Overlap: 0
Val-Test Overlap: 0


# Data Augmentation

Variables

In [11]:
img_size = (300,300)
batch_size = 32

In [12]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,

    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.10,
    zoom_range=0.25,

    horizontal_flip=True,

    brightness_range=[0.7, 1.3],

    fill_mode='nearest'
)

test_val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

In [13]:
train_batches = train_datagen.flow_from_directory(
    train_dir, target_size=img_size, batch_size=batch_size,
    class_mode='binary', shuffle=True, seed=SEED
)
val_batches = test_val_datagen.flow_from_directory(
    val_dir, target_size=img_size, batch_size=batch_size,
    class_mode='binary', shuffle=True, seed=SEED
)
test_batches = test_val_datagen.flow_from_directory(
    test_dir, target_size=img_size, batch_size=batch_size,
    class_mode='binary', shuffle=False, seed=SEED
)

Found 33434 images belonging to 2 classes.
Found 4179 images belonging to 2 classes.
Found 4180 images belonging to 2 classes.


In [14]:
print("\n--- Class Distribution ---")
for name, batches in zip(['Train', 'Validation', 'Test'],
                         [train_batches, val_batches, test_batches]):
    counts = Counter(batches.classes)
    print(f"{name}: {dict(counts)}")


--- Class Distribution ---
Train: {np.int32(0): 17878, np.int32(1): 15556}
Validation: {np.int32(0): 2235, np.int32(1): 1944}
Test: {np.int32(0): 2235, np.int32(1): 1945}


# **Neural Network *(NN)***

Helper Methods

In [15]:
def conv_fn(x, filters, kernel, strides=1, padding='same'):
    """Conv2D + BatchNormalization + ReLU, returns the direct output."""
    x = Conv2D(filters, kernel, strides=strides, padding=padding, use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    return x

Inception Customized Methods

In [16]:
def reduction_A_maker_fn(x):
    branch1 = MaxPooling2D((3, 3), strides=2, padding='valid')(x)

    branch2 = Conv2D(384, 3, strides=2, padding='valid', use_bias=True)(x)
    branch2 = BatchNormalization()(branch2)
    branch2 = Activation('relu')(branch2)

    branch3 = conv_fn(x, 256, 1, padding='same')
    branch3 = conv_fn(branch3, 256, 3, padding='same')
    branch3 = Conv2D(384, 3, strides=2, padding='valid', use_bias=True)(branch3)
    branch3 = BatchNormalization()(branch3)
    branch3 = Activation('relu')(branch3)

    return Concatenate()([branch1, branch2, branch3])

def inception_resnet_B_maker_fn(x):
    in_channels = x.shape[-1]
    b1 = conv_fn(x, 192, 1)
    b2 = conv_fn(x, 128, 1)
    b2 = conv_fn(b2, 160, (1, 7))
    b2 = conv_fn(b2, 192, (7, 1))
    mixed = Concatenate()([b1, b2])
    mixed = Conv2D(in_channels, 1, padding='same', use_bias=True)(mixed)
    mixed = BatchNormalization()(mixed)
    return Activation('relu')(Add()([x, mixed]))

def reduction_B_maker_fn(x):
    b1 = MaxPooling2D((3, 3), strides=2, padding='valid')(x)

    b2 = conv_fn(x, 256, 1)
    b2 = Conv2D(384, 3, strides=2, padding='valid', use_bias=True)(b2)
    b2 = BatchNormalization()(b2)
    b2 = Activation('relu')(b2)

    b3 = conv_fn(x, 256, 1)
    b3 = Conv2D(256, 3, strides=2, padding='valid', use_bias=True)(b3)
    b3 = BatchNormalization()(b3)
    b3 = Activation('relu')(b3)

    b4 = conv_fn(x, 256, 1)
    b4 = conv_fn(b4, 256, 3, padding='same')
    b4 = Conv2D(256, 3, strides=2, padding='valid', use_bias=True)(b4)
    b4 = BatchNormalization()(b4)
    b4 = Activation('relu')(b4)

    return Concatenate()([b1, b2, b3, b4])


Build Hybrid Model

In [17]:
def build_model(input_shape=(300, 300, 3)):
    inputs = Input(shape=input_shape)
    backbone = EfficientNetB3(include_top=False, weights='imagenet', input_tensor=inputs)
    x = backbone.output

    # Add custom blocks
    x = reduction_A_maker_fn(x)
    x = inception_resnet_B_maker_fn(x)

    x = reduction_B_maker_fn(x)

    # Classification head
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.4)(x)
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs, outputs)
    return model, backbone

In [18]:
model, backbone = build_model()
backbone.trainable = False

43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


# **Train**

In [19]:
class_counts = Counter(train_batches.classes)
total = sum(class_counts.values())
class_weight = {
    0: total / (2 * class_counts[0]),
    1: total / (2 * class_counts[1])
}
print(f"\nClass weights: {class_weight}")


Class weights: {0: 0.935059850095089, 1: 1.07463358189766}


In [20]:
model.compile(
    optimizer=Adam(learning_rate=0.0003),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc')
    ]
)

callbacks_phase1 = [
    EarlyStopping(monitor='val_auc', patience=6, mode='max', restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_auc', factor=0.3, patience=4, mode='max', min_lr=1e-6),
    ModelCheckpoint('best_model_phase1.h5', monitor='val_auc', mode='max',
                    save_best_only=True, verbose=1)
]

Phase 1: Training the Head

In [21]:
history1 = model.fit(
    train_batches,
    validation_data=val_batches,
    epochs=15,
    class_weight=class_weight,
    callbacks=callbacks_phase1
)

Epoch 1/15
1045/1045 ━━━━━━━━━━━━━━━━━━━━ 0s 935ms/step - accuracy: 0.8665 - auc: 0.9273 - loss: 0.3435 - precision: 0.8510 - recall: 0.8605
Epoch 1: val_auc improved from None to 0.99997, saving model to best_model_phase1.h5



Epoch 1: finished saving model to best_model_phase1.h5
1045/1045 ━━━━━━━━━━━━━━━━━━━━ 1112s 992ms/step - accuracy: 0.9361 - auc: 0.9836 - loss: 0.1637 - precision: 0.9279 - recall: 0.9354 - val_accuracy: 0.9986 - val_auc: 1.0000 - val_loss: 0.0057 - val_precision: 0.9985 - val_recall: 0.9985 - learning_rate: 3.0000e-04
Epoch 2/15
 468/1045 ━━━━━━━━━━━━━━━━━━━━ 8:34 892ms/step - accuracy: 0.9824 - auc: 0.9973 - loss: 0.0548 - precision: 0.9808 - recall: 0.9814

KeyboardInterrupt: 

Phase 2: Fine-Tuning

In [ ]:
model = keras.models.load_model('best_model_phase1.h5', compile=False)

for layer in model.layers:
    layer.trainable = False

for layer in model.layers:
    if layer.name.startswith("block6") or layer.name.startswith("top") or layer.name.startswith("dense"):
        layer.trainable = True
        print(f"Layer {layer.name} is trainable")

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc')
    ]
)

callbacks_phase2 = [
    EarlyStopping(monitor='val_auc', patience=8, mode='max', restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_auc', factor=0.3, patience=5, mode='max', min_lr=1e-7),
    ModelCheckpoint('best_model_finetuned.h5', monitor='val_auc', mode='max',
                    save_best_only=True, verbose=1)
]

In [ ]:
history2 = model.fit(
    train_batches,
    validation_data=val_batches,
    epochs=12,
    class_weight=class_weight,
    callbacks=callbacks_phase2
)

In [ ]:
model = keras.models.load_model('best_model_finetuned.h5')

# Plot Training Curves

In [ ]:
def plot_training(history, title):
    metrics = ['loss', 'accuracy', 'precision', 'recall', 'auc']
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()
    for i, metric in enumerate(metrics):
        ax = axes[i]
        ax.plot(history.history[metric], label='Train')
        ax.plot(history.history[f'val_{metric}'], label='Val')
        ax.set_title(f'{metric.upper()} - {title}')
        ax.legend()
    # Hide the sixth (empty) subplot
    axes[-1].axis('off')
    plt.tight_layout()
    plt.show()

plot_training(history1, 'Phase 1')
plot_training(history2, 'Phase 2 (Fine-Tuning)')

# Evaluate

In [ ]:
print("\n--- Evaluation on Test Set ---")
test_loss, test_acc, test_prec, test_rec, test_auc = model.evaluate(test_batches, verbose=0)
print(f"Loss: {test_loss:.4f}, Acc: {test_acc:.4f}, Prec: {test_prec:.4f}, Rec: {test_rec:.4f}, AUC: {test_auc:.4f}")

y_pred_probs = model.predict(test_batches).flatten()
y_true = test_batches.classes


# Threshold Tuning

In [ ]:
val_probs = model.predict(val_batches).flatten()
val_true = val_batches.classes

precisions, recalls, thresholds = precision_recall_curve(val_true, val_probs)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_thresh = thresholds[np.argmax(f1_scores)]
print(f"\nOptimal threshold based on F1-score (validation set): {best_thresh:.4f}")

y_pred_opt = (y_pred_probs > best_thresh).astype(int)

print("\n--- Classification Report (Default Threshold 0.5) ---")
print(classification_report(y_true, (y_pred_probs > 0.5).astype(int),
                            target_names=['Non Drowsy', 'Drowsy']))

print("\n--- Classification Report (Optimized Threshold) ---")
print(classification_report(y_true, y_pred_opt,
                            target_names=['Non Drowsy', 'Drowsy']))

# Plot confusion matrix
plt.figure(figsize=(5,4))
sns.heatmap(confusion_matrix(y_true, y_pred_opt), annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non Drowsy', 'Drowsy'],
            yticklabels=['Non Drowsy', 'Drowsy'])
plt.title('Confusion Matrix (Optimized Threshold)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

# Test-Time Augmentation (TTA)

In [ ]:
def predict_with_tta(model, directory, target_size, batch_size, n_aug=5):
    """
    Predict using an average of n_aug random augmentations on the test data.
    Note: The directory must contain the same subfolder structure as the training data.
    """
    tta_datagen = ImageDataGenerator(
        rescale=1./255,
        horizontal_flip=True,
        rotation_range=10,
        zoom_range=0.1,
        brightness_range=[0.9, 1.1]
    )
    all_probs = []
    for i in range(n_aug):
        print(f"TTA round {i+1}/{n_aug}")
        gen = tta_datagen.flow_from_directory(
            directory,
            target_size=target_size,
            batch_size=batch_size,
            class_mode=None,      # we don't need labels
            shuffle=False
        )
        probs = model.predict(gen)
        all_probs.append(probs.flatten())
    return np.mean(all_probs, axis=0)


In [ ]:
print("\n--- Applying Test-Time Augmentation ---")
y_pred_tta_probs = predict_with_tta(model, test_dir, img_size, batch_size, n_aug=5)
y_pred_tta = (y_pred_tta_probs > best_thresh).astype(int)
print(classification_report(y_true, y_pred_tta))